# 채보 규칙 분석 특징 다시 저장 — GPU 서버 실행 (S15P31A307-164)

GMD 45개·MDB 23곡마다 채보 모델(ADTOF)의 프레임별 확신 점수와 드럼 킷 분리 트랙 6개를 계산해 `data/rule_features/`에 저장한다. 이 파일만 있으면 규칙 비교는 로컬 CPU로 몇 초에 끝난다. 셀마다 명령만 부르고 로직은 없다. **위에서부터 차례로 실행한다.**

**준비물.** 로컬 `AI/`에서 만든 zip 4개와 이 노트북을 JupyterHub `ssaplash/upload/` 폴더에, 노트북은 `ssaplash/notebooks/`에 올린다(파일 목록 화면 오른쪽 위 Upload). 서버 폴더 구성: `ssaplash/upload`(올린 zip) · `ssaplash/S15P31A307`(코드) · `ssaplash/datasets`(큰 원본 데이터) · `ssaplash/results`(내려받을 결과 zip) · `ssaplash/notebooks`.

| 파일 | 내용 | 크기 |
| --- | --- | --- |
| `S15P31A307_code.zip` | 저장소 코드. **이번 브랜치 코드로 새로 만든 것**이어야 한다 | 약 2MB |
| `data_gmd.zip` | GMD test 연주 45개 | 약 550MB |
| `data_mdb_drums.zip` | MDB Drums 23곡 | 약 350MB |
| `models_drumsep.zip` | 드럼 킷 분리 가중치 | 약 420MB |

**순서.** 1·2번 준비(1~2분), 3번 설치(처음 한 번 10~20분), 4번 본작업(H200 기준 10~20분), 5번 결과 묶기. 셀 하나가 끝나면 마지막 줄을 확인하고 다음 셀로 간다.

**끊기면.** 브라우저를 닫아도 셀은 끝까지 돈다. 커널이 다시 시작됐으면 1번 셀만 다시 실행하고 끊긴 셀부터 이어 간다. 4번은 `--overwrite`라 처음부터 다시 돈다.

In [ ]:
# 1. 설정. GPU_ID에 이 서버에서 지정받은 GPU 번호를 넣음 (공용 서버라 지정된 GPU만 씀)
import os
import sys

GPU_ID = ""  # 예: "3"
assert GPU_ID.strip(), "GPU_ID에 지정받은 GPU 번호를 넣고 다시 실행"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONUTF8"] = "1"
os.environ["MPLBACKEND"] = "Agg"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
PY = sys.executable
UV = f"{PY} -m uv"
UPLOAD = f"{WORK}/upload"  # 올린 zip
RESULTS = f"{WORK}/results"  # 내려받을 결과 zip
os.makedirs(RESULTS, exist_ok=True)
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!df -h $HOME | tail -1
!ls -la $UPLOAD/*.zip
# 확인: GPU 한 줄이 찍히고, upload 폴더에 zip 4개가 보여야 함. 디스크 여유는 10GB 이상

In [ ]:
# 2. 코드·데이터 풀기. 코드는 새로 올린 zip으로 항상 다시 풂(예전 YourMT3 때 코드가 남아 있을 수 있음). 데이터는 이미 있으면 건너뜀
import shutil
old_features = f"{AI}/data/rule_features"
keep = f"{WORK}/_keep_rule_features"
if os.path.isdir(old_features):  # 예전에 만든 특징이 있으면 잠깐 치워 둠 (코드 폴더를 지우므로)
    shutil.move(old_features, keep)
if os.path.isdir(f"{WORK}/S15P31A307"):
    shutil.rmtree(f"{WORK}/S15P31A307")
!$PY -m zipfile -e $UPLOAD/S15P31A307_code.zip $WORK
for name, folder in [
    ("data_gmd", "data/gmd"),
    ("data_mdb_drums", "data/mdb_drums/MDB Drums"),
    ("models_drumsep", "data/models/mdx23c_drumsep"),
]:
    if not os.path.isdir(f"{AI}/{folder}") and os.path.isfile(f"{UPLOAD}/{name}.zip"):
        !$PY -m zipfile -e $UPLOAD/{name}.zip $AI
%cd $AI
!git -C .. log --oneline -1 2>/dev/null || true
!du -sh data/gmd data/mdb_drums data/models 2>/dev/null
!ls src/ssaplash_ai/bench/analysis/
# 확인: data/gmd·data/mdb_drums·data/models 세 줄이 찍히고 analysis 폴더에 rule_features.py 가 있어야 함

In [ ]:
# 3. 환경. 이번엔 분리 모델(models)과 채보 모델(transcribers) extra가 꼭 필요함.
#    extra를 깔면 torch가 CUDA 13판으로 들어오는데 서버 드라이버는 CUDA 12.8이라, 그 뒤에 CUDA 12.8판 torch를 덮어씀. 순서를 바꾸면 안 됨.
#    이후 명령은 uv run 대신 .venv/bin/ssaplash-bench 로 부름 (uv run은 잠금 파일대로 torch를 되돌림)
# uv sync가 Python 3.11을 못 받으면: conda create -n py311 python=3.11 -y 뒤 $UV sync --python ~/.conda/envs/py311/bin/python ...
# pip --user가 막히면: curl -LsSf https://astral.sh/uv/install.sh | sh 로 ~/.local/bin/uv 를 깔고 UV = f"{HOME}/.local/bin/uv"
!$PY -m pip install --user -q uv
!$UV --version
!$UV sync --extra models --extra transcribers
!$UV pip install --python .venv/bin/python torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!.venv/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"
!.venv/bin/python -c "import adtof_pytorch, msst; print('adtof ok, msst ok')"
# 확인: 'cuda True NVIDIA H200 NVL' 과 'adtof ok, msst ok'. cuda False면 멈춤(CPU로 돌면 몇 시간 걸림).
# adtof 줄에서 오류가 나면 서버가 GitHub에 못 붙은 것 → 로컬 .venv/Lib/site-packages/adtof_pytorch 폴더를 zip으로 올려 .venv/lib/python3.11/site-packages/ 에 품

In [ ]:
# 4. 특징 저장. MDB 23곡 먼저(5분 안팎), 그다음 GMD 45개. --overwrite 라 옛 파일이 있어도 전부 다시 만듦.
#    곡마다 '탐 N·심벌 M (초)' 한 줄이 찍힘. 분리 가중치가 data/models 에 없으면 GitHub에서 내려받음
!.venv/bin/ssaplash-bench rule-features --dataset configs/bench/datasets/mdb_drums.yaml --overwrite
!.venv/bin/ssaplash-bench rule-features --dataset configs/bench/datasets/gmd_test.yaml --overwrite
!ls data/rule_features/mdb_drums | wc -l
!ls data/rule_features/gmd_test | wc -l
!.venv/bin/python -c "import numpy as np; z=np.load('data/rule_features/mdb_drums/MusicDelta_Grunge.npz'); print(sorted(z.files))"
# 확인: 23 과 45. 마지막 줄에 'activations' 와 'stem_kick'·'stem_snare'·'stem_hh' 가 보여야 함. 실패 곡이 있으면 그 곡 이름과 오류를 적어 둠

In [ ]:
# 5. 결과 묶기. results 폴더의 rule_features.zip 을 파일 목록 화면에서 내려받아 로컬 AI/data/rule_features/ 를 비우고 거기에 품
!rm -f $RESULTS/rule_features.zip
!$PY -m zipfile -c $RESULTS/rule_features.zip data/rule_features
!ls -la $RESULTS/rule_features.zip
# 확인: 2~3GB. 내려받은 뒤 로컬 AI 폴더에서 python -m zipfile -e rule_features.zip . 으로 품